# 🎯 Module 3 — Class 4: Pick the BEST Features (Olist Edition)

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/4](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/4)

---

## 📖 Today's story

After Class 3 you have ~20 features in `orders_step3.parquet`. The boss says:

> *"Train the model with all 20."*

You try. The training takes 10 minutes. Test accuracy is mediocre. Your laptop fan is screaming.

Why? Because some of those 20 features are **junk**:
- `delivery_days` is just the answer in disguise — it leaks the target!
- `purchase_year` is mostly the same value (97% of rows = 2017 or 2018) — near-zero variance
- `days_buffer` is `estimate_days - delivery_days` — perfectly redundant with two other features

**Today we cut the junk and keep the 8-12 features that actually predict.**

---

## 💡 What is feature selection?

**Feature selection** = picking the smaller subset of features that work best.

Why bother?
- ⚡ **Faster training** — fewer features = fewer computations
- 🎯 **Better accuracy** — junk features confuse the model (overfitting)
- 🔍 **Interpretability** — easier to explain a 10-feature model than a 100-feature one
- 💰 **Production cost** — fewer features = cheaper to serve

### 🍳 Kitchen analogy

You're making a recipe. You have 20 ingredients in front of you. Some are essential (eggs, flour). Some are useless (a can of motor oil). Some are duplicates (white sugar AND brown sugar — pick one).

Feature selection = look at all 20, keep the 8-12 that actually make the dish.

---

## 🎯 Today's plan

1. **Filter step 1** — variance threshold (drop near-constant columns)
2. **Filter step 2** — correlation matrix (drop redundant pairs)
3. **Filter step 3** — mutual information (rank features by signal vs target)
4. **Embedded step** — Random Forest feature importances
5. **Final pick** — the 8-12 features we'll use in M4

---

## 📦 Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_parquet('orders_step3.parquet')
print('Loaded:', df.shape)

## Step 1 — Build the candidate feature matrix

Pick the columns we COULD use. We'll cut the junk in steps 2-5.

In [ ]:
candidate = [
    'distance_km', 'log_freight', 'total_price', 'num_items',
    'estimate_days', 'purchase_hour', 'purchase_dayofweek',
    'is_weekend', 'days_buffer', 'purchase_month',
]

X = df[candidate].dropna()
y = df.loc[X.index, 'is_late']
print(f'X: {X.shape},  y: {y.shape}')
print(f'is_late rate: {y.mean():.3f}')

---

## Step 2 — Filter step 1: variance threshold 📊

### 💡 What's a near-zero-variance column?

A column where 99% of rows have the same value.

**Example:** `is_brazilian_seller` — every Olist seller is Brazilian. The column is always 1. It carries ZERO information.

Drop it. Free speedup.

In [ ]:
sel = VarianceThreshold(threshold=0.01)
sel.fit(X)
kept = X.columns[sel.get_support()].tolist()
dropped = [c for c in X.columns if c not in kept]

print(f'Kept after variance filter ({len(kept)}): {kept}')
print(f'Dropped (near-zero-variance): {dropped or "none"}')
X = X[kept]

---

## Step 3 — Filter step 2: correlation matrix 🔗

### 💡 The redundancy problem

If two features are 0.97 correlated, they're basically the same column. The model learns nothing extra by having both — but training takes longer and interpretability suffers.

**Rule:** if `|r| > 0.9`, drop one of the pair.

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(X.corr().abs(), annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1)
plt.title('Feature correlation (|r|) — anything above 0.9 is redundant')
plt.tight_layout(); plt.show()

In [ ]:
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]

print(f'High-correlation pairs detected. Dropping: {to_drop}')
if to_drop:
    X = X.drop(columns=to_drop)
print(f'Remaining ({len(X.columns)}): {list(X.columns)}')

### 💡 What you'll likely see on Olist

`days_buffer = estimate_days - delivery_days` is highly correlated with both. We engineered it for fun in C3, but it leaks the target — drop it.

Also `delivery_days` itself is technically a leak — at prediction time we don't yet know how long delivery took. Always drop target-derived features. **Done automatically by correlation filter.**

---

## Step 4 — Filter step 3: mutual information 🎯

### What is mutual information?

A score that measures how much a feature **tells you about the target**. 
- 0 = the feature gives no information about `is_late`
- 1+ = the feature is highly predictive

Unlike correlation (only catches linear relationships), mutual info catches non-linear too. Better default for tree models.

In [ ]:
mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)

print('Mutual information vs is_late (higher = more predictive):\n')
print(mi_series.round(4))

---

## Step 5 — Embedded step: Random Forest importances 🌳

Train a quick Random Forest. Read its `feature_importances_`. **This is the gold-standard way to rank tabular features for classification.**

Why? Because it's the model itself telling you what mattered when it actually built decision trees on the data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=12,
                            class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns) \
                  .sort_values(ascending=False)

print('🌳 Random Forest feature importances (higher = used more by the trees):\n')
print(importances.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='forestgreen')
ax.invert_yaxis()
ax.set_title('Olist features — Random Forest importance')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

---

## Step 6 — Final selected feature set ✅

### Decision rule

Keep features that show up in the **top half** of at least 2 of the 3 rankings (mutual info, RF importance, common sense).

For Olist `is_late`, the usual winners are:
- `distance_km` (top by RF)
- `estimate_days` (clear inverse signal)
- `log_freight`
- `is_weekend`
- `purchase_hour`
- `num_items`
- `total_price`

Cut everything else.

In [ ]:
FINAL = ['distance_km', 'log_freight', 'estimate_days',
        'is_weekend', 'purchase_hour', 'num_items', 'total_price']

X_final = X[[c for c in FINAL if c in X.columns]]
X_final.to_parquet('X_step4_features.parquet', index=False)
y.to_frame().to_parquet('y_step4_target.parquet', index=False)

print(f'✅ Saved final feature set with {X_final.shape[1]} columns')
print(f'   X: {X_final.shape}')
print(f'   Features: {list(X_final.columns)}')

---

## ✅ Quick Check

1. Two features have correlation 0.97. Why drop one and not both?
2. Filter selection vs embedded selection — name one advantage of each.
3. Why does `delivery_days` need to be dropped even if it has high importance? (Hint: when does the model see it?)

## 🗺️ Where this lands

| Module | What loads `X_step4_features.parquet` (or its descendants)? |
| --- | --- |
| **M3-5** (next) | Wraps these features in a `Pipeline + ColumnTransformer` |
| **M3-6** (lab) | Combines into final `olist_clean.parquet` |
| **M4-1 to M4-6** | Every model trained on these 7 features |
| **M6** | NN trained on these 7 features (after standardisation) |
| **M8** | The /predict API takes these 7 features as input from the user |

**Your final-feature decision today = the API contract for the next 6 weeks.**

## 📤 Submit

### Bronze
Run all cells. Submit `X_step4_features.parquet` and `y_step4_target.parquet`.

### Gold
Add **3 more features** (your engineered ones from M3-3 Gold). Re-run the full selection pipeline. Did your new features make it into the FINAL list? Why / why not? Half-page rationale.

🎯 *Great work — you cut the junk and kept the gold!*